In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
# Load dataset
df = read_delta("final_features")
df = df.drop(["product_name"])

In [ ]:
# Encode categorical columns
for col in ["department", "aisle"]:
    encoder = LabelEncoder()
    encoded = encoder.fit_transform(df[col].to_list())
    df = df.with_columns(pl.Series(name=col, values=encoded))

In [ ]:
# Define hypothesis variables
X = df.drop(["user_id", "product_id", "reordered"])
y = df["reordered"]

In [ ]:
# Create dataset splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# Calculate class imbalance
neg, pos = (df["reordered"] == 0).sum(), (df["reordered"] == 1).sum()
ratio = neg / pos

In [ ]:
# Initialise Aim run
run = new_run(experiment="xgb-reorder-predictor")
run["hparams"] = {
    "scale_pos_weight": ratio,
    "n_estimators": 100,
    "learning_rate": 0.1,
    "max_depth": 6,
}

In [ ]:
# Define model
model = xgb.XGBClassifier(
    objective="binary:logistic",
    scale_pos_weight=ratio,
    eval_metric="logloss",
    use_label_encoder=False,
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

In [ ]:
# Train model
eval_set = [("train", X_train, y_train), ("val", X_test, y_test)]
model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)

In [ ]:
# Retroactively log metrics to Aim
results = model.evals_result()
train_losses = results["validation_0"]["logloss"]
val_losses = results["validation_1"]["logloss"]

for i, (train_loss, val_loss) in enumerate(zip(train_losses, val_losses)):
    run.track(train_loss, name="train_loss", step=i)
    run.track(val_loss, name="val_loss", step=i)

In [ ]:
# Final evaluation
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("Accuracy:", acc)
print(classification_report(y_test, y_pred))

run.track(acc, name="final_accuracy")